In [27]:
import pandas as pd
import numpy as np
import requests
import time
import mygene
import myvariant

from Bio import Entrez, SeqIO

In [2]:
df = pd.read_csv("datosGene4PD/t_common_variant.txt", sep = "\t", index_col = False)

In [3]:
df

,Chr,gene_symbol,SNPs_symbol,SNP_position,effect_allele,alternate_allele,joint_phase_P,joint_phase_OR,joint_phase_OR_CI,pubMed_ID,Unnamed: 10
0,6,GPR126,rs757765789,142758601,T,G,1.42E-06,1.06,1.016-1.098,28256260,NaN
1,1,SYT11,rs202015799,155839054,C,T,4.70E-09,-,-,24842889,NaN
2,12,SLC2A13,rs1994090,40428561,G,T,3.20E-54,12.05,8.35-17.41,24842889,NaN
3,12,SLC2A13,rs2708453,40478652,G,T,3.62E-54,12.05,8.35-17.41,24842889,NaN
4,12,SLC2A13,rs4768212,40474147,C,T,3.62E-54,12.05,8.35-17.41,24842889,NaN
...,...,...,...,...,...,...,...,...,...,...,...
1053,rs117896735,INPP5F,13,U,13,U,1.21E-11,1.77,-,29700661,NaN
1054,rs12456492,RIT2,21,U,21,U,2.15E-11,1.1,-,29700661,NaN
1055,rs7155501,GCH1,15,U,15,U,1.25E-10,1.12,-,29700661,NaN
1056,rs10797576,SIPA1L2,21,U,21,U,1.76E-10,1.13,-,29700661,NaN


In [4]:
df_filtrado = df[df["SNPs_symbol"].str.startswith('rs')].reset_index(drop = True)

In [5]:
df_filtrado = df_filtrado.drop(["Unnamed: 10"], axis = 1)

In [6]:
df_filtrado = df_filtrado.dropna(subset = ['gene_symbol']).reset_index(drop = True)

In [21]:
mv = myvariant.MyVariantInfo()

In [22]:
prueba = mv.querymany(['rs757765789'], scopes = "dbsnp.rsid", fields = "dbsnp", species = "human")

In [27]:
prueba[0]["dbsnp"]["gene"]["strand"]

'+'

In [9]:
def busca_rsIDs_cadena(df):
    
    lista_rsids = df["SNPs_symbol"].unique().tolist()

    mv = myvariant.MyVariantInfo()

    resultados = mv.querymany(lista_rsids, scopes = "dbsnp.rsid", fields = "dbsnp", species = "human")

    diccionario_snps = {}

    cromosomas_validos = [str(i) for i in range(1, 23)] + ["X", "Y", "MT"]

    bases_validas = ["A", "C", "G", "T", "a", "c", "g", "t"]

    cadenas_validas = ["+", "-", "1", "-1"]

    for resultado in resultados:
        rsid = resultado.get("query")

        if "notfound" in resultado:
            continue

        info_dbsnp = resultado.get("dbsnp", {})

        hg19 = info_dbsnp.get("hg19", {})

        if not hg19:
            continue

        if isinstance(hg19, list):
            hg19 = hg19[0]

        cromosoma = str(info_dbsnp.get("chrom"))
        if cromosoma not in cromosomas_validos:
            continue

        pos_hg19 = hg19.get("start")
    
        ef_allele = info_dbsnp.get("ref", '')
        alt_allele = info_dbsnp.get("alt", '')

        if ef_allele not in bases_validas or alt_allele not in bases_validas:
            continue

        gene_info = info_dbsnp.get("gene", {})
        gene_symbol = "inter"

        if isinstance(gene_info, dict):
            gene_symbol = gene_info.get("symbol", "inter")
            cadena = gene_info.get("strand")

        elif isinstance(gene_info, list):
            gene_symbol = gene_info[0].get("symbol", "inter")
            cadena = gene_info[0].get("strand")

        if rsid not in diccionario_snps:

            diccionario_snps[rsid] = {"SNPs_symbol": rsid, "Chr_corr": cromosoma, "SNP_position_corr": pos_hg19, "Effect_allele_corr": ef_allele, "Alternate_allele_corr": [alt_allele], "Gene_symbol_corr": gene_symbol, "Cadena": cadena}

        else:

            if alt_allele not in diccionario_snps[rsid]["Alternate_allele_corr"]:
                diccionario_snps[rsid]["Alternate_allele_corr"].append(alt_allele)

    datos_corregidos = list(diccionario_snps.values())

    return pd.DataFrame(datos_corregidos)

In [10]:
df_corr = busca_rsIDs(df_filtrado)

474 input query terms found dup hits:	[('rs202015799', 2), ('rs1994090', 3), ('rs2708453', 2), ('rs4768212', 2), ('rs7304281', 2), ('rs204
1 input query terms found no hit:	['rs2740594c']


In [11]:
df_corr

,SNPs_symbol,Chr_corr,SNP_position_corr,Effect_allele_corr,Alternate_allele_corr,Gene_symbol_corr,Cadena
0,rs757765789,6,142758601,T,[G],ADGRG6,+
1,rs202015799,1,155839054,C,"[G, T]",SYT11,+
2,rs1994090,12,40428561,G,"[A, T, C]",SLC2A13,-
3,rs2708453,12,40478652,G,"[A, T]",SLC2A13,-
4,rs4768212,12,40474147,C,"[A, T]",SLC2A13,-
...,...,...,...,...,...,...,...
881,rs666463,17,76425480,A,[T],DNAH17,-
882,rs1941685,18,31304318,G,"[T, C]",ASXL3,+
883,rs8087969,18,48683589,T,"[G, A]",inter,None
884,rs77351827,20,6006041,C,[T],CRLS1,+


In [12]:
df_nuevo = pd.merge(df_filtrado, df_corr, on = "SNPs_symbol", how = "left")

In [13]:
df_nuevo

,Chr,gene_symbol,SNPs_symbol,SNP_position,effect_allele,alternate_allele,joint_phase_P,joint_phase_OR,joint_phase_OR_CI,pubMed_ID,Chr_corr,SNP_position_corr,Effect_allele_corr,Alternate_allele_corr,Gene_symbol_corr,Cadena
0,6,GPR126,rs757765789,142758601,T,G,1.42E-06,1.06,1.016-1.098,28256260,6,142758601.0,T,[G],ADGRG6,+
1,1,SYT11,rs202015799,155839054,C,T,4.70E-09,-,-,24842889,1,155839054.0,C,"[G, T]",SYT11,+
2,12,SLC2A13,rs1994090,40428561,G,T,3.20E-54,12.05,8.35-17.41,24842889,12,40428561.0,G,"[A, T, C]",SLC2A13,-
3,12,SLC2A13,rs2708453,40478652,G,T,3.62E-54,12.05,8.35-17.41,24842889,12,40478652.0,G,"[A, T]",SLC2A13,-
4,12,SLC2A13,rs4768212,40474147,C,T,3.62E-54,12.05,8.35-17.41,24842889,12,40474147.0,C,"[A, T]",SLC2A13,-
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1009,17,DNAH17,rs666463,76425480,A,T,2.90E-09,1.08,-,https://doi.org/10.1101/388165,17,76425480.0,A,[T],DNAH17,-
1010,18,ASXL3,rs1941685,31304318,T,G,1.61E-08,1.05,-,https://doi.org/10.1101/388165,18,31304318.0,G,"[T, C]",ASXL3,+
1011,18,MEX3C,rs8087969,48683589,T,G,1.46E-08,0.94,-,https://doi.org/10.1101/388165,18,48683589.0,T,"[G, A]",inter,None
1012,20,CRLS1,rs77351827,6006041,T,C,7.94E-09,1.08,-,https://doi.org/10.1101/388165,20,6006041.0,C,[T],CRLS1,+


In [15]:
df_nuevo = df_nuevo[["Chr", "SNPs_symbol", "SNP_position", "effect_allele", "alternate_allele", "Chr_corr", "SNP_position_corr", "Effect_allele_corr", "Alternate_allele_corr", "Cadena"]]

In [16]:
df_nuevo

,Chr,SNPs_symbol,SNP_position,effect_allele,alternate_allele,Chr_corr,SNP_position_corr,Effect_allele_corr,Alternate_allele_corr,Cadena
0,6,rs757765789,142758601,T,G,6,142758601.0,T,[G],+
1,1,rs202015799,155839054,C,T,1,155839054.0,C,"[G, T]",+
2,12,rs1994090,40428561,G,T,12,40428561.0,G,"[A, T, C]",-
3,12,rs2708453,40478652,G,T,12,40478652.0,G,"[A, T]",-
4,12,rs4768212,40474147,C,T,12,40474147.0,C,"[A, T]",-
...,...,...,...,...,...,...,...,...,...,...
1009,17,rs666463,76425480,A,T,17,76425480.0,A,[T],-
1010,18,rs1941685,31304318,T,G,18,31304318.0,G,"[T, C]",+
1011,18,rs8087969,48683589,T,G,18,48683589.0,T,"[G, A]",None
1012,20,rs77351827,6006041,T,C,20,6006041.0,C,[T],+


In [17]:
df_puro = df_nuevo.dropna(subset = ["SNP_position_corr"]).reset_index(drop = True)
df_puro = df_puro.dropna(subset = ["Cadena"]).reset_index(drop = True)

In [18]:
df_puro

,Chr,SNPs_symbol,SNP_position,effect_allele,alternate_allele,Chr_corr,SNP_position_corr,Effect_allele_corr,Alternate_allele_corr,Cadena
0,6,rs757765789,142758601,T,G,6,142758601.0,T,[G],+
1,1,rs202015799,155839054,C,T,1,155839054.0,C,"[G, T]",+
2,12,rs1994090,40428561,G,T,12,40428561.0,G,"[A, T, C]",-
3,12,rs2708453,40478652,G,T,12,40478652.0,G,"[A, T]",-
4,12,rs4768212,40474147,C,T,12,40474147.0,C,"[A, T]",-
...,...,...,...,...,...,...,...,...,...,...
770,17,rs61169879,59917366,T,C,17,59917366.0,C,"[A, T]",-
771,17,rs666463,76425480,A,T,17,76425480.0,A,[T],-
772,18,rs1941685,31304318,T,G,18,31304318.0,G,"[T, C]",+
773,20,rs77351827,6006041,T,C,20,6006041.0,C,[T],+


In [19]:
df_puro["SNP_position_corr"] = df_puro["SNP_position_corr"].astype(int)

In [20]:
def valida_alelos_SNP(fila):

    effect_allele_original = fila["effect_allele"]
    alternate_allele_original = fila["alternate_allele"]
    effect_allele_rsID = fila["Effect_allele_corr"]
    alternate_allele_rsID = fila["Alternate_allele_corr"]

    alelos_original = [effect_allele_original, alternate_allele_original]
    alelos_rsID = [effect_allele_rsID] + alternate_allele_rsID

    if all(alelo in alelos_rsID for alelo in alelos_original):
        
        return "Coincide perfecto"

    complementario = {"A": "T", "T": "A", "C": "G", "G": "C"}
    alelos_original_complementario = [complementario.get(alelo, alelo) for alelo in alelos_original]

    if all(alelo in alelos_rsID for alelo in alelos_original_complementario):
        
        return "Coincide complementario"

    return "No coincide"

In [21]:
df_puro["Valida_alelos"] = df_puro.apply(valida_alelos_SNP, axis = 1)

In [22]:
print(df_puro["Valida_alelos"].value_counts())

Coincide perfecto          762
No coincide                 10
Coincide complementario      3
Name: Valida_alelos, dtype: int64


In [23]:
df_puro = df_puro[df_puro["Valida_alelos"] != "No coincide"].reset_index(drop = True)

In [24]:
complementario = {"A": "T", "T": "A", "C": "G", "G": "C"}

In [25]:
filas_comp = df_puro["Valida_alelos"] == "Coincide complementario"
df_puro.loc[filas_comp, "effect_allele"] = df_puro.loc[filas_comp, "effect_allele"].map(complementario)
df_puro.loc[filas_comp, "alternate_allele"] = df_puro.loc[filas_comp, "alternate_allele"].map(complementario)

In [26]:
df_puro

,Chr,SNPs_symbol,SNP_position,effect_allele,alternate_allele,Chr_corr,SNP_position_corr,Effect_allele_corr,Alternate_allele_corr,Cadena,Valida_alelos
0,6,rs757765789,142758601,T,G,6,142758601,T,[G],+,Coincide perfecto
1,1,rs202015799,155839054,C,T,1,155839054,C,"[G, T]",+,Coincide perfecto
2,12,rs1994090,40428561,G,T,12,40428561,G,"[A, T, C]",-,Coincide perfecto
3,12,rs2708453,40478652,G,T,12,40478652,G,"[A, T]",-,Coincide perfecto
4,12,rs4768212,40474147,C,T,12,40474147,C,"[A, T]",-,Coincide perfecto
...,...,...,...,...,...,...,...,...,...,...,...
760,17,rs61169879,59917366,T,C,17,59917366,C,"[A, T]",-,Coincide perfecto
761,17,rs666463,76425480,A,T,17,76425480,A,[T],-,Coincide perfecto
762,18,rs1941685,31304318,T,G,18,31304318,G,"[T, C]",+,Coincide perfecto
763,20,rs77351827,6006041,T,C,20,6006041,C,[T],+,Coincide perfecto


In [29]:
archivo_genoma = "datosGene4PD/Homo_sapiens.GRCh37.completo.fa"

genoma = SeqIO.index(archivo_genoma, "fasta")

In [ ]:
def extrae_region_sana(df_final_fila, ventana):

    cromosoma = df_final_fila["Chr_corr"]
    cadena = df_final_fila["cadena"]
    pos_snp = df_final_fila["SNP_position_corr"]

    alelo_sano = df_final_fila["effect_allele"]

    secuencia_chr = genoma[cromosoma].seq

    inicio = max(0, pos_snp - ventana)
    fin = pos_snp + (ventana + 1)

    pos_relativa_snp = pos_snp - inicio

    reg_sana_forward = secuencia_chr[inicio : fin]
    

    if cadena == 1:
        
        reg_sana = reg_sana_forward

    else:

        reg_sana = str(Seq(reg_sana_forward).reverse_complement())

    return reg_sana

In [ ]:
def extrae_region_parkinson(df_final_fila, reg_sana):
    
    # reg_sana = str(extrae_region_sana(df_final_fila))
    cadena = df_final_fila["cadena"]
    inicio = df_final_fila["inicio"]
    fin = df_final_fila["fin"]
    pos_snp = df_final_fila["SNP_position_corr"]

    alelo_parkinson = df_final_fila["alternate_allele"]
    

    if cadena == 1:

        snp = pos_snp - inicio
        nuc = alelo_parkinson
    
        reg_park = reg_sana[:snp] + nuc + reg_sana[snp + 1:]

    else:

        snp = fin - pos_snp
        nuc_comp = str(Seq(alelo_parkinson).complement())

        reg_park = reg_sana[:snp] + nuc_comp + reg_sana[snp + 1:]

    return reg_park